# 🎯 Hyperparameter Tuning con Amazon SageMaker

## 📋 ¿Qué hace esta notebook?

Esta notebook demuestra cómo realizar **ajuste automático de hiperparámetros (Hyperparameter Tuning)** usando Amazon SageMaker. El objetivo es encontrar la mejor combinación de hiperparámetros para un modelo de clasificación de vinos.

### 🔍 Flujo del proceso:

1. **Carga de datos**: Descargamos un dataset de calidad de vinos tintos
2. **Preprocesamiento**: Dividimos los datos en conjuntos de entrenamiento, validación y prueba
3. **Almacenamiento**: Subimos los datos a Amazon S3
4. **Configuración del modelo**: Creamos un estimador de XGBoost en SageMaker
5. **Hyperparameter Tuning**: Definimos rangos de hiperparámetros y ejecutamos múltiples entrenamientos
6. **Análisis de resultados**: Identificamos el mejor modelo y lo preparamos para despliegue

### 🎮 ¿Qué puedes modificar para experimentar?

#### Hyperparámetros a ajustar:
- **alpha**: Regularización L1 (0-1000) - Mayor valor = más penalización
- **eta**: Tasa de aprendizaje (0.1-0.5) - Controla la velocidad de aprendizaje
- **min_child_weight**: Peso mínimo para divisiones (1-120) - Controla la profundidad del árbol
- **subsample**: Fracción de datos por árbol (0.5-1) - Previene overfitting
- **num_round**: Número de árboles (1-4000) - Más árboles = modelo más complejo

#### Configuración de SageMaker:
- **max_jobs**: Número total de combinaciones a probar (actualmente: 40)
- **max_parallel_jobs**: Entrenamientos en paralelo (actualmente: 2)
- **instance_type**: Tipo de instancia EC2 (actualmente: ml.m5.xlarge)
- **objective_metric**: Métrica a optimizar (actualmente: validation:mlogloss)

---

In [ ]:
# 📦 Importación de librerías necesarias

# Librerías para manejo de datos
import pandas as pd
from sklearn.model_selection import train_test_split

# Librerías de AWS
import boto3  # SDK de AWS para Python
import io
import os
import uuid
from datetime import datetime

# Librerías de SageMaker
from sagemaker import get_execution_role
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput
from sagemaker.parameter import (
    CategoricalParameter,  # Para hiperparámetros categóricos (ej: 'gbtree', 'dart')
    ContinuousParameter,   # Para hiperparámetros continuos (ej: 0.1 - 0.5)
    IntegerParameter       # Para hiperparámetros enteros (ej: 1 - 4000)
)
from sagemaker.tuner import HyperparameterTuner
from sagemaker.estimator import Estimator

## 📊 Paso 1: Carga y preprocesamiento de datos

**¿Qué hace este paso?**
- Descarga el dataset de vinos tintos desde UCI Machine Learning Repository
- Mapea las etiquetas de calidad (3-8) a clases numéricas (0-5) para clasificación
- Reorganiza las columnas para que 'quality' sea la primera (requerido por XGBoost)

**💡 Puedes experimentar con:**
- Diferentes datasets de UCI: https://archive.ics.uci.edu/ml/datasets.php
- Cambiar el mapeo de clases (agregar más o menos categorías)
- Usar tus propios datos en formato CSV

In [ ]:
# Cargar el dataset de calidad de vinos tintos
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df_wine = pd.read_csv(url, sep=';')  # El delimitador es punto y coma (;)
df_wine.head()

# Mapear la columna 'quality' a clases numéricas (0-5)
# Esto convierte las calidades originales (3, 4, 5, 6, 7, 8) en clases consecutivas
df_wine['quality'] = df_wine['quality'].map({3: 0, 4: 1, 5: 2, 6: 3, 7: 4, 8: 5})

# Reordenar columnas para poner 'quality' primero
# XGBoost requiere que la variable objetivo esté en la primera columna
cols = df_wine.columns.tolist()
cols = cols[-1:] + cols[:-1]  # Mueve la última columna al inicio
df_wine = df_wine[cols]

print(f"✅ Dataset cargado: {len(df_wine)} filas, {len(df_wine.columns)} columnas")
print(f"📈 Distribución de clases:\n{df_wine['quality'].value_counts().sort_index()}")

## ✂️ Paso 2: División de datos (Train/Validation/Test)

**¿Qué hace este paso?**
- Divide los datos en 3 conjuntos:
  - **Train (80%)**: Para entrenar el modelo
  - **Validation (10%)**: Para evaluar durante el tuning
  - **Test (10%)**: Para evaluación final (no usado en tuning)
- Usa `stratify` para mantener la proporción de clases en cada conjunto

**💡 Puedes experimentar con:**
- Cambiar las proporciones: `test_size=0.3` (30% test, 70% train)
- Modificar `random_state=42` para obtener splits diferentes
- Eliminar `stratify` si no te importa el balance de clases

In [ ]:
# Primera división: 80% train, 20% test+validation
train, test_and_validate = train_test_split(
    df_wine, 
    test_size=0.2,           # 20% para test+validation
    random_state=42,         # Semilla para reproducibilidad
    stratify=df_wine['quality']  # Mantiene proporción de clases
)

# Segunda división: dividir test+validation en 50-50
test, validate = train_test_split(
    test_and_validate, 
    test_size=0.5,           # 50% para test, 50% para validation
    random_state=42,
    stratify=test_and_validate['quality']
)

print(f"📊 Tamaños de los conjuntos:")
print(f"  - Train: {len(train)} filas ({len(train)/len(df_wine)*100:.1f}%)")
print(f"  - Validation: {len(validate)} filas ({len(validate)/len(df_wine)*100:.1f}%)")
print(f"  - Test: {len(test)} filas ({len(test)/len(df_wine)*100:.1f}%)")

## ☁️ Paso 3: Subir datos a Amazon S3

**¿Qué hace este paso?**
- Configura un bucket de S3 para almacenar los datos
- Verifica si el bucket existe o lo crea
- Sube los conjuntos de train y validation como archivos CSV
- SageMaker leerá estos datos desde S3 durante el entrenamiento

**💡 Puedes experimentar con:**
- Cambiar el prefijo: `prefix = 'mi-proyecto-ml'`
- Agregar el conjunto de test: `upload_s3_csv('wine_test.csv', 'test', test)`
- Usar diferentes formatos: Parquet, JSON, etc.

In [ ]:
# Configuración del bucket de S3
bucket='c188735a4871751l12674432t1w119436913223-labbucket-up1ntvkr4pen'
prefix = 'wine-hyperparam'  # Carpeta dentro del bucket
region = boto3.Session().region_name
s3_resource = boto3.resource('s3', region_name=region)

# Crear el bucket si no existe
try:
    s3_resource.meta.client.head_bucket(Bucket=bucket)
    print(f"✅ Bucket '{bucket}' ya existe.")
except Exception:
    # Crear bucket según la región
    if region == "us-east-1":
        s3_resource.create_bucket(Bucket=bucket)
    else:
        s3_resource.create_bucket(
            Bucket=bucket,
            CreateBucketConfiguration={'LocationConstraint': region}
        )
    print(f"✅ Bucket '{bucket}' creado exitosamente.")

# Función para subir DataFrames como CSV a S3
def upload_s3_csv(filename, folder, dataframe):
    """
    Sube un DataFrame de pandas como CSV a S3
    
    Args:
        filename: Nombre del archivo (ej: 'wine_train.csv')
        folder: Carpeta dentro del prefijo (ej: 'train')
        dataframe: DataFrame de pandas a subir
    """
    csv_buffer = io.StringIO()
    # Guardar sin headers ni índices (formato requerido por XGBoost)
    dataframe.to_csv(csv_buffer, header=False, index=False)
    
    # Subir a S3
    s3_path = os.path.join(prefix, folder, filename)
    s3_resource.Bucket(bucket).Object(s3_path).put(Body=csv_buffer.getvalue())
    print(f"📤 Subido: s3://{bucket}/{s3_path}")

# Subir los datasets a S3
upload_s3_csv('wine_train.csv', 'train', train)
upload_s3_csv('wine_validate.csv', 'validate', validate)

## 🤖 Paso 4: Crear el Estimator de SageMaker

**¿Qué hace este paso?**
- Crea un estimator de XGBoost usando la imagen de contenedor de SageMaker
- Configura los recursos de cómputo (tipo de instancia, cantidad)
- Define la ubicación de salida en S3 para los modelos entrenados
- Establece hiperparámetros base que NO serán tuneados

**💡 Puedes experimentar con:**

### Instance types (tipos de instancia):
- `ml.m5.xlarge`: 4 vCPUs, 16 GB RAM - Opción económica ✅
- `ml.m5.2xlarge`: 8 vCPUs, 32 GB RAM - Más potente
- `ml.c5.2xlarge`: 8 vCPUs, 16 GB RAM - Optimizado para cómputo
- `ml.p3.2xlarge`: Con GPU (más costoso) - Para datasets grandes

### Hiperparámetros fijos:
- `num_round`: Número de árboles (actualmente: 40)
- `num_class`: Número de clases (6 en este caso: 0-5)
- `objective`: Tipo de problema ('multi:softmax' para clasificación multiclase)

In [ ]:
import sagemaker

# Crear una sesión de SageMaker
sagemaker_session = sagemaker.Session()

# Obtener la imagen de contenedor de XGBoost
# Version 1.0-1 es estable y probada
container = sagemaker.image_uris.retrieve(
    'xgboost',  # Algoritmo
    region=boto3.Session().region_name,
    version='1.0-1'  # Versión del contenedor
)

# Obtener el rol de ejecución de SageMaker
# Este rol debe tener permisos para acceder a S3 y otros servicios
role = sagemaker.get_execution_role()

# Definir la ubicación de salida en S3
s3_output_location = f"s3://{bucket}/{prefix}/output/"

# Crear el estimator de XGBoost
xgb_model = sagemaker.estimator.Estimator(
    image_uri=container,              # Imagen de contenedor de XGBoost
    role=role,                        # Rol IAM con permisos
    instance_count=1,                 # Número de instancias (1 para empezar)
    instance_type='ml.m5.xlarge',     # 🎮 CAMBIA ESTO para más potencia
    output_path=s3_output_location,   # Donde guardar los modelos
    sagemaker_session=sagemaker_session
)

# Configurar hiperparámetros FIJOS (no serán tuneados)
xgb_model.set_hyperparameters(
    num_round=40,               # 🎮 Número de árboles base
    num_class=6,                # Número de clases (0-5)
    objective='multi:softmax'   # Tipo de problema: clasificación multiclase
)

print(f"✅ Estimator creado con éxito")
print(f"📦 Contenedor: {container}")
print(f"💻 Tipo de instancia: ml.m5.xlarge")
print(f"📁 Salida: {s3_output_location}")


## 🎯 Paso 5: Definir y ejecutar el trabajo de Hyperparameter Tuning

**¿Qué hace este paso?**
- Define los rangos de hiperparámetros a explorar
- Configura la métrica objetivo a optimizar (validation:mlogloss)
- Crea un HyperparameterTuner que ejecutará múltiples entrenamientos
- Inicia el proceso de tuning (puede tardar 30-60 minutos)

**💡 Aquí está la MAGIA - Puedes experimentar con:**

### 🔧 Rangos de hiperparámetros:
```python
'alpha': ContinuousParameter(0, 1000)     # 🎮 Regularización L1
'eta': ContinuousParameter(0.1, 0.5)      # 🎮 Learning rate (0.01-1.0)
'min_child_weight': ContinuousParameter(1, 120)  # 🎮 Peso mínimo (1-300)
'subsample': ContinuousParameter(0.5, 1)  # 🎮 Muestra de datos (0.5-1.0)
'num_round': IntegerParameter(1, 4000)    # 🎮 Número de árboles (10-10000)
```

### 📊 Configuración del Tuner:
- **max_jobs**: Total de combinaciones a probar (10-100) - Más jobs = mejor resultado pero más costoso
- **max_parallel_jobs**: Jobs en paralelo (1-10) - Más paralelo = más rápido pero más costoso
- **objective_type**: 'Minimize' o 'Maximize' según la métrica

### 📈 Métricas alternativas:
- `validation:accuracy` - Precisión (Maximize)
- `validation:auc` - Área bajo la curva (Maximize)
- `validation:f1` - F1 Score (Maximize)
- `validation:mlogloss` - Log loss (Minimize) ✅

In [ ]:
# ═══════════════════════════════════════════════════════════
# 🎮 EXPERIMENTA AQUÍ: Define los rangos de hiperparámetros
# ═══════════════════════════════════════════════════════════

hyperparameter_ranges = {
    # Regularización L1: Mayor valor = más penalización, menos overfitting
    'alpha': ContinuousParameter(0, 1000),  
    
    # Learning rate: Controla qué tan rápido aprende el modelo
    # Valores bajos (0.01-0.1) = más estable pero lento
    # Valores altos (0.3-0.5) = más rápido pero puede no converger
    'eta': ContinuousParameter(0.1, 0.5),  
    
    # Peso mínimo para crear una nueva hoja en el árbol
    # Valores altos = árboles más conservadores
    'min_child_weight': ContinuousParameter(1, 120),  
    
    # Fracción de datos usados para entrenar cada árbol
    # Valores < 1 ayudan a prevenir overfitting
    'subsample': ContinuousParameter(0.5, 1),  
    
    # Número de árboles (rondas de boosting)
    # Más árboles = modelo más complejo
    'num_round': IntegerParameter(1, 4000)  
}

# ═══════════════════════════════════════════════════════════
# 🎯 EXPERIMENTA AQUÍ: Define la métrica objetivo
# ═══════════════════════════════════════════════════════════

objective_metric_name = 'validation:mlogloss'  # 🎮 CAMBIA ESTO: mlogloss, accuracy, auc, f1
objective_type = 'Minimize'  # 🎮 CAMBIA ESTO: Minimize o Maximize

# ═══════════════════════════════════════════════════════════
# 🚀 Crear y ejecutar el Hyperparameter Tuner
# ═══════════════════════════════════════════════════════════

tuner = HyperparameterTuner(
    estimator=xgb_model,                          # Modelo base configurado
    objective_metric_name=objective_metric_name,  # Métrica a optimizar
    hyperparameter_ranges=hyperparameter_ranges,  # Rangos definidos arriba
    max_jobs=40,                                  # 🎮 Total de entrenamientos (10-100)
    max_parallel_jobs=2,                          # 🎮 Entrenamientos en paralelo (1-10)
    objective_type=objective_type,                # Minimizar o maximizar
    strategy='Bayesian'                           # Estrategia: 'Bayesian' (inteligente) o 'Random'
)

print("🚀 Iniciando Hyperparameter Tuning...")
print(f"📊 Se probarán {tuner.max_jobs} combinaciones")
print(f"⚡ En paralelo: {tuner.max_parallel_jobs} jobs")
print(f"🎯 Optimizando: {objective_metric_name} ({objective_type})")
print("\n⏰ Esto puede tardar 30-60 minutos. Puedes:")
print("   1. Cerrar la notebook (el job seguirá corriendo)")
print("   2. Ver el progreso en la consola de SageMaker")
print("   3. Ejecutar la siguiente celda para esperar aquí")

# Iniciar el tuning
tuner.fit(inputs={
    'train': TrainingInput(f"s3://{bucket}/{prefix}/train/", content_type='text/csv'),
    'validation': TrainingInput(f"s3://{bucket}/{prefix}/validate/", content_type='text/csv')
})

# Esperar a que termine (opcional - puedes comentar esta línea)
tuner.wait()

print("\n✅ ¡Hyperparameter Tuning completado!")

In [ ]:
# 🔍 Verificar el estado del trabajo de tuning
# Esto te permite ver si el job está corriendo, completado o falló

status = boto3.client('sagemaker').describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuner.latest_tuning_job.job_name
)['HyperParameterTuningJobStatus']

print(f"📊 Estado del job: {status}")
print(f"🏷️ Nombre del job: {tuner.latest_tuning_job.job_name}")

# Estados posibles:
# - InProgress: Ejecutándose
# - Completed: Finalizado con éxito
# - Failed: Falló
# - Stopping: Deteniéndose
# - Stopped: Detenido manualmente

In [ ]:
# 📈 Analizar los resultados del Hyperparameter Tuning
# Esta celda crea un DataFrame con todos los entrenamientos realizados

from pprint import pprint
from sagemaker.analytics import HyperparameterTuningJobAnalytics

# Crear un objeto para analizar los resultados
tuner_analytics = HyperparameterTuningJobAnalytics(
    tuner.latest_tuning_job.name,
    sagemaker_session=sagemaker.Session()
)

# Convertir los resultados en un DataFrame de pandas
df_tuning_job_analytics = tuner_analytics.dataframe()

# Ordenar por el valor de la métrica objetivo
# Si minimizamos (mlogloss), queremos los valores más bajos primero
# Si maximizamos (accuracy), queremos los valores más altos primero
df_tuning_job_analytics.sort_values(
    by=['FinalObjectiveValue'],
    inplace=True,
    ascending=False if tuner.objective_type == "Maximize" else True
)

print(f"🎯 Mejor resultado: {df_tuning_job_analytics.iloc[0]['FinalObjectiveValue']:.6f}")
print(f"📊 Total de entrenamientos: {len(df_tuning_job_analytics)}")
print("\n🏆 Top 5 mejores modelos:")

# Mostrar los 5 mejores modelos con sus hiperparámetros
df_tuning_job_analytics.head()

In [ ]:
# 🏆 Obtener el mejor modelo del tuning
# El tuner automáticamente identifica cuál fue el mejor entrenamiento

# Adjuntar el tuner al trabajo de tuning más reciente
# (útil si cerraste la notebook y quieres recuperar los resultados)
attached_tuner = HyperparameterTuner.attach(
    tuner.latest_tuning_job.name,
    sagemaker_session=sagemaker.Session()
)

# Obtener el nombre del mejor trabajo de entrenamiento
best_training_job = attached_tuner.best_training_job()

print(f"🏆 Mejor modelo: {best_training_job}")
print("\n💡 TIP: Este es el job que usaremos para crear el modelo final")

In [ ]:
# 📦 Crear un modelo de SageMaker a partir del mejor entrenamiento
# Este modelo puede ser desplegado como un endpoint para hacer predicciones

from sagemaker.estimator import Estimator

# Adjuntar el estimator al mejor trabajo de entrenamiento
algo_estimator = Estimator.attach(best_training_job)

# Crear un modelo de SageMaker
# Este modelo puede ser desplegado posteriormente
best_algo_model = algo_estimator.create_model(
    env={'SAGEMAKER_DEFAULT_INVOCATIONS_ACCEPT': "text/csv"}  # Formato de entrada: CSV
)

print("✅ Modelo creado exitosamente")
print(f"📦 Nombre del modelo: {best_algo_model.name}")
print("\n🚀 Próximos pasos posibles:")
print("   1. Desplegar el modelo: best_algo_model.deploy()")
print("   2. Hacer predicciones batch")
print("   3. Evaluar en el conjunto de test")
print("   4. Exportar el modelo para uso local")

## 🎮 Guía de Experimentación: ¿Qué puedes cambiar?

### 1️⃣ Hiperparámetros de XGBoost (Paso 5)

#### **alpha** - Regularización L1
```python
'alpha': ContinuousParameter(0, 1000)  # Prueba: (0, 100) o (0, 5000)
```
- Controla la penalización L1 en los pesos
- **Valores altos (>500)**: Más regularización, menos overfitting
- **Valores bajos (<100)**: Menos regularización, puede overfit

#### **eta** - Learning Rate
```python
'eta': ContinuousParameter(0.1, 0.5)  # Prueba: (0.01, 0.3) o (0.3, 0.9)
```
- Controla qué tan rápido aprende el modelo
- **Valores bajos (0.01-0.1)**: Aprendizaje lento pero estable
- **Valores altos (0.3-0.5)**: Aprendizaje rápido pero puede ser inestable

#### **min_child_weight** - Peso mínimo
```python
'min_child_weight': ContinuousParameter(1, 120)  # Prueba: (1, 50) o (10, 300)
```
- Suma mínima de pesos para crear una nueva hoja
- **Valores altos (>50)**: Árboles conservadores, menos overfitting
- **Valores bajos (<10)**: Árboles más profundos, puede overfit

#### **subsample** - Fracción de datos
```python
'subsample': ContinuousParameter(0.5, 1)  # Prueba: (0.6, 0.9) o (0.3, 1.0)
```
- Fracción de datos usados para entrenar cada árbol
- **Valores bajos (<0.7)**: Más variación, previene overfitting
- **Valores altos (>0.9)**: Usa más datos, más estable

#### **num_round** - Número de árboles
```python
'num_round': IntegerParameter(1, 4000)  # Prueba: (10, 1000) o (100, 10000)
```
- Número de árboles (rondas de boosting)
- **Valores altos (>1000)**: Modelo más complejo, puede overfit
- **Valores bajos (<100)**: Modelo simple, puede underfit

---

### 2️⃣ Configuración del Tuner (Paso 5)

#### **max_jobs** - Total de entrenamientos
```python
max_jobs=40  # Prueba: 10, 20, 50, 100
```
- **Menos jobs (10-20)**: Más rápido, menos costoso, puede no encontrar el óptimo
- **Más jobs (50-100)**: Más lento, más costoso, mejor probabilidad de encontrar el óptimo

#### **max_parallel_jobs** - Entrenamientos en paralelo
```python
max_parallel_jobs=2  # Prueba: 1, 5, 10
```
- **Menos paralelo (1-2)**: Más lento, menos costoso
- **Más paralelo (5-10)**: Más rápido, más costoso

#### **strategy** - Estrategia de búsqueda
```python
strategy='Bayesian'  # Prueba: 'Random' o 'Bayesian'
```
- **Bayesian**: Inteligente, aprende de resultados previos (recomendado)
- **Random**: Aleatorio, útil para exploración inicial

---

### 3️⃣ Métricas Objetivo (Paso 5)

```python
objective_metric_name = 'validation:mlogloss'  # Cambia esto
objective_type = 'Minimize'  # Cambia según la métrica
```

| Métrica | Tipo | Descripción |
|---------|------|-------------|
| `validation:mlogloss` | Minimize | Log loss multiclase (actual) ✅ |
| `validation:accuracy` | Maximize | Precisión (% correctos) |
| `validation:merror` | Minimize | Error de clasificación |
| `train:mlogloss` | Minimize | Log loss en train (cuidado: overfit) |

---

### 4️⃣ Tipos de Instancia (Paso 4)

```python
instance_type='ml.m5.xlarge'  # Cambia esto
```

| Tipo | vCPUs | RAM | Costo relativo | Uso recomendado |
|------|-------|-----|----------------|------------------|
| `ml.m5.large` | 2 | 8 GB | $ | Desarrollo/testing |
| `ml.m5.xlarge` | 4 | 16 GB | $$ | Datos pequeños ✅ |
| `ml.m5.2xlarge` | 8 | 32 GB | $$$ | Datos medianos |
| `ml.c5.4xlarge` | 16 | 32 GB | $$$$ | CPU intensivo |
| `ml.p3.2xlarge` | 8 | 61 GB | $$$$$ | GPU, datasets grandes |

---

### 5️⃣ División de Datos (Paso 2)

```python
test_size=0.2  # Prueba: 0.1, 0.3, 0.4
```
- **Más training (0.9)**: Modelo ve más datos, mejor entrenamiento
- **Más test (0.3-0.4)**: Mejor evaluación, menos datos para entrenar

---

### 💡 Tips para experimentar:

1. **Empieza simple**: Usa `max_jobs=10` para probar rápido
2. **Verifica overfitting**: Compara train vs validation metrics
3. **Ajusta rangos**: Si todos los mejores modelos tienen eta=0.5, amplía el rango
4. **Monitorea costos**: Más jobs y paralelo = más costoso
5. **Usa Bayesian**: Es más eficiente que Random para explorar

---

### 🚀 Experimentos sugeridos:

#### Experimento 1: Búsqueda rápida
```python
max_jobs=10
max_parallel_jobs=2
eta: (0.1, 0.5)
num_round: (50, 500)
```

#### Experimento 2: Búsqueda exhaustiva
```python
max_jobs=100
max_parallel_jobs=10
eta: (0.01, 0.9)
num_round: (10, 5000)
```

#### Experimento 3: Optimizar regularización
```python
max_jobs=30
alpha: (0, 2000)
min_child_weight: (1, 300)
subsample: (0.3, 1.0)
```